__Importing Libraries__

In [ ]:

from pyspark.sql import SparkSession
from pyspark.sql.functions import *


In [ ]:
#Creating a SparkSession
spark=SparkSession.builder.appName('Customer Data Analysis').getOrCreate()
spark

## Reading data from Different Sources 

__Reading CSV file__

In [ ]:
%%bash
head -10 ./Data/Superstore.csv

In [ ]:
#Read CSV file with headers

df=spark.read.csv("Data/Superstore.csv",header=True)
df.show(5)

In [ ]:
df.printSchema()

In [ ]:
df=spark.read.csv("Data/Superstore.csv",header=True,inferSchema=True,quote='"',escape='"',multiLine=True)
df.printSchema()

In [ ]:
try:
    df.write.parquet("Data/Superstore.parquet")
    print("Parquet formated file Successfully saved")
except :
    print("File is already Saved")

In [ ]:
#Reading Data into parquet format

df=spark.read.parquet("Data/Superstore.parquet")

print("Structure of the Data:")
df.printSchema()

print("--Printing the Data--")
df.show(4)

In [ ]:
df = df.withColumnRenamed("Sales", "Price")


date_cols = ["Ship Date", "Order Date"]
for d in date_cols:
    df = df.withColumn(d, to_date(col(d), "M/d/yyyy"))

In [ ]:
df.printSchema()

__Transformation__

In [ ]:
selected_column = (df
                    .select('Order ID','Customer ID','Product ID','Price','Quantity','Category')
                    .withColumn("Total_amount", df.Price * df.Quantity)

)
selected_column = selected_column.filter(selected_column.Total_amount > 50000)
selected_column.show()


In [ ]:
selected_column=selected_column.groupBy('Category').count()
selected_column.show()

__Saving/Writing the Data__

In [ ]:
df.write.mode("overwrite").parquet("output")
print("Data saved Successfully")

## Summary

### Pipeline Steps
| Step | Operation | Method |
|---|---|---|
| 1 | Load CSV | `spark.read.csv` with quote/escape handling |
| 2 | Convert to Parquet | `df.write.parquet` — columnar, faster reads |
| 3 | Rename + Cast | `withColumnRenamed`, `cast('double')`, `to_date` |
| 4 | Filter (AND) | `(col('Total_amount') > 50000)` |
| 5 | Derive column | `withColumn('total_amount', Price * quantity)` |
| 6 | Aggregate | `groupBy('Category').agg(count)` |
| 7 | Save output | Parquet  using `write.mode('overwrite')` |

### Key Insights
- **Parquet vs CSV:** Parquet is columnar — queries reading only 2-3 columns skip the rest entirely, reducing I/O significantly on large datasets
- **Lazy Evaluation:** Transformations like `filter()`, `withColumn()` build a DAG but execute only when an action like `.show()` or `.count()` is called
- **Predicate Pushdown:** When reading Parquet, Spark pushes filter conditions into the file reader — only matching row groups are loaded into memory
- **Avoid collect():** On large datasets `collect()` brings all data to the driver — use `.show(n)` instead
- **Wide vs Narrow:** `groupBy()` is a wide transformation causing shuffle — data moves across partitions. `filter()` and `select()` are narrow — no shuffle needed
